In [104]:
from langchain_mineru import MinerULoader
import pymupdf

import fitz


from colpali_engine.models import ColQwen2_5, ColQwen2_5_Processor
import torch
from PIL import Image
from transformers.utils.import_utils import is_flash_attn_2_available

from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path # Will add a function to check the path and shi later

ModuleNotFoundError: No module named 'colpali_engine'

# importing the paper and checking for the path of the paper for further convertion and loading

In [97]:
paper_path = "../data/paper/attention_is_all_you_need.pdf"
output_path = Path("../data/parsed")
copali_image_path = Path("../data/parsed/paper_images")

In [99]:
copali_image_path.mkdir(parents=True, exist_ok=True)
# paper_path.mkdir(parents=True, exist_ok=True)
# output_path.mkdir(parents=True, exist_ok=True)

# Loading the paper with minerU

In [38]:
loader = MinerULoader(source=paper_path, split_pages=True) # Need to keep the split pages True because imma use copali later that'll analyze the pdf page by page for visual elements
docs = loader.load()

In [45]:
print(docs[3])

page_content='<!-- image-->  
Figure 2: (left) Scaled Dot-Product Attention. (right) Multi-Head Attention consists of several attention layers running in parallel.

query with all keys, divide each by $\sqrt { d _ { k } } ,$ and apply a softmax function to obtain the weights on the values.

In practice, we compute the attention function on a set of queries simultaneously, packed together into a matrix Q. The keys and values are also packed together into matrices K and V . We compute the matrix of outputs as:

$$
\mathrm { A t t e n t i o n } ( Q , K , V ) = \mathrm { s o f t m a x } ( \frac { Q K ^ { T } } { \sqrt { d _ { k } } } ) V\tag{1}
$$

The two most commonly used attention functions are additive attention [2], and dot-product (multiplicative) attention. Dot-product attention is identical to our algorithm, except for the scaling factor of $\frac { 1 } { \sqrt { d _ { k } } }$ . Additive attention computes the compatibility function using a feed-forward network with a single hidd

In [39]:
print(len(docs))

11


In [40]:
for i, doc in enumerate(docs[:3]):
    print(f"Document {i}")
    print(f"Metadata: {doc.metadata}")
    print(f"Content:\n{doc.page_content[:1000]}")

Document 0
Metadata: {'source': '../data/paper/attention_is_all_you_need.pdf', 'loader': 'mineru', 'output_format': 'markdown', 'mode': 'flash', 'language': 'ch', 'pages': None, 'split_pages': True, 'filename': None, 'page': 1, 'page_source': '../data/paper/attention_is_all_you_need.pdf'}
Content:
# Attention Is All You Need

Ashish Vaswani∗ Google Brain avaswani@google.com

Noam Shazeer∗ Google Brain noam@google.com

Niki Parmar∗   
Google Research   
nikip@google.com

Jakob Uszkoreit∗ Google Research usz@google.com

Llion Jones∗ Google Research llion@google.com

Aidan N. Gomez∗ † University of Toronto aidan@cs.toronto.edu

Łukasz Kaiser∗ Google Brain lukaszkaiser@google.com

Illia Polosukhin∗ ‡illia.polosukhin@gmail.com

## Abstract

The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a 

In [59]:
for i, doc in enumerate(docs, start=1):
    page_path = output_path / f"page_{i:03d}.md" # Three zeroes before the page number

    page_path.write_text(
        doc.page_content,
    )

print(f"Saved {len(docs)} pages to {output_path}")

Saved 11 pages to ../data/parsed


# Processing for copali

In [92]:
zoom = 2.0  
matrix = fitz.Matrix(zoom, zoom)

In [90]:
paper = pymupdf.open(str(paper_path))

In [91]:
page_1 = paper.load_page(1)
page_1.number
# pix = page_1.get_pixmap()
# pix.save(f"page-{page.number}.png")

1

In [95]:
for i, page in enumerate(paper, start=1):
    # Pass the matrix to render at a higher resolution
    pix = page.get_pixmap(matrix=matrix)
    image_output_path = copali_image_path / f"page_{i:03d}.png"

    pix.save(str(image_output_path))
    pix=None # To free up the memory by letting go the previosu one

print("saved")




saved


# Copali: Utilizing a vision encoder in order to convert the images into mathematical feature vectors, actally multi vector representation

In [100]:
model_name = "vidore/colqwen2-v1.0"

In [ ]:
model = ColQwen2.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="cuda:0" 
    attn_implementation="flash_attention_2" if is_flash_attn_2_available() else None,
).eval()

processor = ColQwen2Processor.from_pretrained(model_name)

NameError: name 'ColQwen2' is not defined